# 04 — Segment Analysis
# 04 — 分群分析

**Objective:** Examine whether the treatment effect differs across user engagement segments.
**目标：** 分析 treatment 效果在不同参与度的用户群体中是否有差异。

Instead of asking only *'did gate_40 affect retention overall?'*, we ask:
*'Which types of users were most affected?'*

我们不只问"gate_40 整体上有没有效果"，而是进一步问：
**"哪类用户受影响最大？对哪类用户几乎没有影响？"**

**Segments (defined in notebook 01) / 分群定义（在 notebook 01 中生成）：**

| Segment | Definition | Description | 中文说明 |
|---|---|---|---|
| `non_player` | 0 rounds | Installed but never played | 装了游戏从没玩过 |
| `low` | 1–4 rounds | Light engagement | 轻度参与，玩了几局就停了 |
| `medium` | 5–29 rounds | Regular players | 普通玩家，有一定参与度 |
| `high` | 30+ rounds | Heavy players | 重度玩家，到达了关卡门槛区域 |

**Input:** `../data/app_ab_test.db`

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest
import warnings
warnings.filterwarnings('ignore')

# Connect to database / 连接数据库
conn = sqlite3.connect('../data/app_ab_test.db')
df = pd.read_sql_query('SELECT * FROM users', conn)

print(f'Loaded {len(df):,} users')
print(f'Segments: {df["engagement_segment"].value_counts().to_dict()}')

Loaded 90,189 users
Segments: {'medium': 35195, 'high': 33269, 'low': 17731, 'non_player': 3994}


---
## 1. D7 Retention Rate by Segment and Group
每个分群的 D7 留存率

In [2]:
# Define segment order for display (from lowest to highest engagement)
# 定义分群的显示顺序（从低参与到高参与）
seg_order = ['non_player', 'low', 'medium', 'high']

results = []

for seg in seg_order:
    # Filter users belonging to this segment
    # 筛选属于这个分群的用户
    seg_df = df[df['engagement_segment'] == seg]

    # Split into control and treatment within this segment
    # 在这个分群内部再分成 control 和 treatment 两组
    ctrl = seg_df[seg_df['version'] == 'gate_30']   # control 组（旧版本）
    trt  = seg_df[seg_df['version'] == 'gate_40']   # treatment 组（新版本）

    n_ctrl = len(ctrl)   # control 组用户数
    n_trt  = len(trt)    # treatment 组用户数

    # Count users retained on Day 7
    # 统计第 7 天留存的用户数
    ret_ctrl = ctrl['retention_7'].sum()   # control 组第7天留存用户数
    ret_trt  = trt['retention_7'].sum()    # treatment 组第7天留存用户数

    # D7 retention rate = retained users / total users
    # D7 留存率 = 第7天留存用户数 / 该分群总用户数
    rate_ctrl = ret_ctrl / n_ctrl if n_ctrl > 0 else 0
    rate_trt  = ret_trt  / n_trt  if n_trt  > 0 else 0

    # Absolute uplift = treatment rate - control rate
    # 绝对提升 = treatment 留存率 - control 留存率
    # Positive = treatment better / 正数 = treatment 更好
    # Negative = treatment worse  / 负数 = treatment 更差
    uplift = rate_trt - rate_ctrl

    results.append({
        'segment':    seg,
        'n_ctrl':     n_ctrl,
        'n_trt':      n_trt,
        'rate_ctrl':  rate_ctrl,
        'rate_trt':   rate_trt,
        'uplift':     uplift,
        'ret_ctrl':   ret_ctrl,
        'ret_trt':    ret_trt,
    })

df_results = pd.DataFrame(results)

# Display results table / 格式化显示结果表格
print('=== D7 Retention by Segment / 各分群 D7 留存率 ===')
print(f'{"Segment":<12} {"n_ctrl":>8} {"n_trt":>8} {"Ctrl D7":>9} {"Trt D7":>9} {"Uplift":>9}')
print('-' * 60)
for _, row in df_results.iterrows():
    print(f'{row["segment"]:<12} {row["n_ctrl"]:>8,} {row["n_trt"]:>8,} '
          f'{row["rate_ctrl"]:>8.1%} {row["rate_trt"]:>8.1%} {row["uplift"]:>+8.2%}')

=== D7 Retention by Segment / 各分群 D7 留存率 ===
Segment        n_ctrl    n_trt   Ctrl D7    Trt D7    Uplift
------------------------------------------------------------
non_player      1,937    2,057     0.8%     0.6%   -0.19%
low             8,677    9,054     1.2%     1.4%   +0.19%
medium         17,430   17,765     6.2%     5.6%   -0.56%
high           16,656   16,613    43.9%    43.0%   -0.87%


---
## Key Observations from Segment Results
## 分群结果的关键发现

**English:**

The segment-level analysis reveals that the treatment effect is **not uniform** across user types:

- **non_player** (0 rounds): Both groups have very low D7 retention (~0.6–0.8%). These users never engaged with the game, so the gate change had no meaningful impact on them.
- **low** (1–4 rounds): The only segment where treatment is *slightly higher* than control (+0.19%). However, the absolute retention rates are very low (1.2% vs 1.4%), so this positive signal is weak.
- **medium** (5–29 rounds): Treatment is -0.56% lower. These are regular players who engaged with the game but did not reach the gate area. The negative effect suggests the new module may have disrupted their experience.
- **high** (30+ rounds): The largest negative effect at -0.87%. These heavy players reached the gate area directly — moving the gate from round 30 to round 40 most likely frustrated them, causing more to drop off before day 7.

**Whether these differences are statistically significant will be tested in the next section.**

---

**中文：**

分群分析表明，treatment 的效果在不同类型用户之间**并不一致**：

- **non_player**（从未玩过）：两组的 D7 留存率都极低（约 0.6–0.8%）。这些用户根本没有参与游戏，关卡门槛的改变对他们没有实质影响。
- **low**（1–4 局）：唯一一个 treatment 略高于 control 的分群（+0.19%）。但两组的留存率本身很低（1.2% vs 1.4%），这个正向信号比较微弱。
- **medium**（5–29 局）：treatment 比 control 低 -0.56%。这类普通玩家有一定参与度，但还没到达关卡门槛区域。负效果说明新模块可能干扰了他们的游戏体验。
- **high**（30+ 局）：负效果最大，-0.87%。这类重度玩家直接接触了关卡门槛——把门槛从第 30 关移到第 40 关，很可能让他们感到沮丧，导致更多人在第 7 天前放弃。

**这些差异是否在统计上显著，将在下一部分通过假设检验来验证。**